# Excitatory Patch-seq ETL

Register VISp excitatory Patch-seq cells, morphology features, T-type mappings, measured MET-type memberships, and inferred MET-type mappings.

In [ ]:
from pathlib import Path

import pandas as pd
import polars as pl
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import write_models
from connects_common_connectivity.io.arrow_utils import build_cell_feature_matrix_schema
from connects_common_connectivity.io.write_utils import walk_ancestors
from connects_common_connectivity.models import (
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    CellToClusterMapping,
    ClusterMembership,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    MappingSet,
    Modality,
)

In [ ]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/EXC Patch-seq"
OUTPUT_ROOT = Path(get_settings().output_root)

PROJECT_ID = "visp_patchseq"
DATASET_ID = "visp_exc_patchseq"
FEATURE_SET_ID = "exc_visp_morph_features"
TTYPE_HIERARCHY_ID = "tasic_2018_visp_taxonomy"
METTYPE_HIERARCHY_ID = "visp_met_types_taxonomy"
TTYPE_MAPPING_SET_ID = "visp_exc_patchseq_ttype_mapping"
INFERRED_MET_MAPPING_SET_ID = "visp_exc_patchseq_inferred_mettype_mapping"
PUBLICATION = "https://doi.org/10.1101/2023.11.25.568393"

ASSIGNMENTS_PATH = DATA_ROOT / "inferred_met_types.csv"
DEFINITIONS_PATH = DATA_ROOT / "exc_visp_patchseq_morph_feature_definitions.csv"
FEATURES_PATH = DATA_ROOT / "morph_features_mMET_exc_wide_unnormalized.csv"
for path in (ASSIGNMENTS_PATH, DEFINITIONS_PATH, FEATURES_PATH):
    assert path.is_file(), f"Missing local input: {path}"

## Dataset and DataItems

In [ ]:
assignment_source = pd.read_csv(ASSIGNMENTS_PATH, index_col=0)
assignment_source.index = assignment_source.index.astype(str)
feature_source = pd.read_csv(FEATURES_PATH).rename(columns={"specimen_id": "id"})
feature_source["id"] = feature_source["id"].astype(str)

# Metadata includes cells without morphology payloads, so register the source union.
registered_ids = set(assignment_source.index) | set(feature_source["id"])
assert registered_ids
dataset = DataSet(
    id=DATASET_ID,
    name="VISp excitatory Patch-seq cells",
    publication=PUBLICATION,
    modality=Modality.MORPHOLOGY,
    project_id=PROJECT_ID,
)
dataitems = [DataItem(id=cell_id, name=cell_id, project_id=PROJECT_ID) for cell_id in sorted(registered_ids)]
associations = [
    DataItemDataSetAssociation(
        dataitem_id=cell_id, dataset_id=DATASET_ID, project_id=PROJECT_ID
    )
    for cell_id in sorted(registered_ids)
]
print(write_models([dataset], output_root=OUTPUT_ROOT))
print(write_models(dataitems, output_root=OUTPUT_ROOT))
print(write_models(associations, output_root=OUTPUT_ROOT))

## Morphology features

In [ ]:
definition_source = pd.read_csv(DEFINITIONS_PATH)
feature_definitions = []
for row in definition_source.to_dict("records"):
    values = {
        "id": str(row["id"]),
        "description": str(row["description"]),
        "unit": str(row["unit"]),
        "data_type": str(row["data_type"]),
        "project_id": PROJECT_ID,
        "feature_set_id": FEATURE_SET_ID,
    }
    if pd.notna(row.get("range_min")):
        values["range_min"] = float(row["range_min"])
    if pd.notna(row.get("range_max")):
        values["range_max"] = float(row["range_max"])
    feature_definitions.append(CellFeatureDefinition(**values))
feature_ids = [definition.id for definition in feature_definitions]
feature_set = CellFeatureSet(
    id=FEATURE_SET_ID,
    description="Unnormalized morphology features for excitatory VISp Patch-seq cells.",
    feature_definition_ids=feature_ids,
    extraction_method="IVSCC morphology feature extraction",
    project_id=PROJECT_ID,
)
assert len(feature_definitions) == 50
assert set(feature_source.columns) == {"id", *feature_ids}
print(write_models(feature_definitions, output_root=OUTPUT_ROOT))
print(write_models([feature_set], output_root=OUTPUT_ROOT))

In [ ]:
# The feature matrix contains values; definitions and the feature set carry schema meaning.
feature_wide = feature_source[["id", *feature_ids]].copy()
feature_wide.insert(1, "project_id", PROJECT_ID)
feature_wide.insert(2, "feature_set_id", FEATURE_SET_ID)
feature_schema = build_cell_feature_matrix_schema(
    feature_set, feature_definitions, cell_index_column="id"
)
feature_table = pa.Table.from_pandas(feature_wide, schema=feature_schema, preserve_index=False)
feature_path = OUTPUT_ROOT / f"cellfeatures/{FEATURE_SET_ID}"
write_deltalake(
    feature_path,
    feature_table,
    mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}' AND feature_set_id = '{FEATURE_SET_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
feature_matrix = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{FEATURE_SET_ID}",
    feature_set_id=FEATURE_SET_ID,
    parquet_path=f"file://{feature_path.resolve()}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
print(write_models([feature_matrix], output_root=OUTPUT_ROOT))

## Cell-type assignments

In [ ]:
cluster_table = pl.read_delta(OUTPUT_ROOT / "cluster")
ttype_clusters = cluster_table.filter(pl.col("hierarchy_id") == TTYPE_HIERARCHY_ID)
mettype_clusters = cluster_table.filter(pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID)
assert not ttype_clusters.is_empty(), "Run the shared T-type taxonomy ETL first."
assert not mettype_clusters.is_empty(), "Run the shared MET-type taxonomy ETL first."
ttype_parent = dict(ttype_clusters.select("id", "parent").iter_rows())
mettype_parent = dict(mettype_clusters.select("id", "parent").iter_rows())

# Source excitatory labels use ET where the shared Tasic hierarchy uses PT.
translated_ttypes = assignment_source["t_type"].map(
    lambda label: label.replace("ET", "PT") if isinstance(label, str) else label
)
unknown_ttypes = set(translated_ttypes.dropna()) - set(ttype_parent)
assert not unknown_ttypes, f"Unknown translated T-types: {sorted(unknown_ttypes)}"
overlap = assignment_source.dropna(subset=["met_type", "inferred_met_type"])
assert overlap["met_type"].eq(overlap["inferred_met_type"]).all()

In [ ]:
ttype_mapping_set = MappingSet(
    id=TTYPE_MAPPING_SET_ID,
    name="VISp excitatory Patch-seq T-type assignments",
    description="Tree mapping of source T-type labels onto the Tasic 2018 VISp taxonomy.",
    method_name="Patch-seq tree mapping",
    source_dataset=DATASET_ID,
    target_hierarchy=TTYPE_HIERARCHY_ID,
    project_id=PROJECT_ID,
)
ttype_mappings = []
for cell_id, leaf in translated_ttypes.dropna().items():
    for cluster_id, is_leaf in walk_ancestors(leaf, ttype_parent):
        ttype_mappings.append(
            CellToClusterMapping(
                id=f"{cell_id}-{cluster_id}-{PROJECT_ID}-{TTYPE_HIERARCHY_ID}",
                mapping_set=TTYPE_MAPPING_SET_ID,
                source_cell=str(cell_id),
                target_cluster=cluster_id,
                project_id=PROJECT_ID,
            )
        )
print(write_models([ttype_mapping_set], output_root=OUTPUT_ROOT))
print(write_models(ttype_mappings, output_root=OUTPUT_ROOT))

In [ ]:
ground_truth = assignment_source.dropna(subset=["met_type"]).copy()
inferred = assignment_source.loc[
    assignment_source["met_type"].isna() & assignment_source["inferred_met_type"].notna()
].copy()
unknown_mettypes = (set(ground_truth["met_type"]) | set(inferred["inferred_met_type"])) - set(mettype_parent)
assert not unknown_mettypes, f"Unknown MET-types: {sorted(unknown_mettypes)}"
assert set(ground_truth.index).isdisjoint(inferred.index)

# Measured MET labels are memberships; inferred labels are explicit mappings.
memberships = []
for cell_id, leaf in ground_truth["met_type"].items():
    for cluster_id, is_leaf in walk_ancestors(leaf, mettype_parent):
        memberships.append(
            ClusterMembership(
                item=str(cell_id), cluster=cluster_id, hierarchy_id=METTYPE_HIERARCHY_ID,
                probability=1.0 if is_leaf else None, project_id=PROJECT_ID,
            )
        )
inferred_mapping_set = MappingSet(
    id=INFERRED_MET_MAPPING_SET_ID,
    name="VISp excitatory Patch-seq inferred MET-type assignments",
    description="Predicted MET-type labels for cells without a measured MET assignment.",
    method_name="Inferred MET-type assignment; source method unspecified",
    source_dataset=DATASET_ID,
    target_hierarchy=METTYPE_HIERARCHY_ID,
    project_id=PROJECT_ID,
)
inferred_mappings = []
for cell_id, leaf in inferred["inferred_met_type"].items():
    for cluster_id, is_leaf in walk_ancestors(leaf, mettype_parent):
        inferred_mappings.append(
            CellToClusterMapping(
                id=f"{cell_id}-{cluster_id}-{PROJECT_ID}-{METTYPE_HIERARCHY_ID}",
                mapping_set=INFERRED_MET_MAPPING_SET_ID,
                source_cell=str(cell_id),
                target_cluster=cluster_id,
                project_id=PROJECT_ID,
            )
        )
print(write_models(memberships, output_root=OUTPUT_ROOT))
print(write_models([inferred_mapping_set], output_root=OUTPUT_ROOT))
print(write_models(inferred_mappings, output_root=OUTPUT_ROOT))

## Verify

In [ ]:
# Verify each scope independently because inhibitory and excitatory cohorts share a project.
stored_associations = pl.read_delta(OUTPUT_ROOT / "dataitem_dataset_association").filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("dataset_id") == DATASET_ID)
)
stored_features = pl.read_delta(feature_path).filter(
    (pl.col("project_id") == PROJECT_ID) & (pl.col("feature_set_id") == FEATURE_SET_ID)
)
all_mappings = pl.read_delta(OUTPUT_ROOT / "celltoclustermapping").filter(
    pl.col("project_id") == PROJECT_ID
)
stored_ttype = all_mappings.filter(pl.col("mapping_set") == TTYPE_MAPPING_SET_ID)
stored_inferred = all_mappings.filter(pl.col("mapping_set") == INFERRED_MET_MAPPING_SET_ID)
stored_memberships = pl.read_delta(OUTPUT_ROOT / "clustermembership").filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("hierarchy_id") == METTYPE_HIERARCHY_ID)
    & pl.col("item").is_in(sorted(set(ground_truth.index)))
)
assert set(stored_associations["dataitem_id"]) == registered_ids
assert set(stored_features["id"]) == set(feature_source["id"])
assert stored_ttype.height == len(ttype_mappings)
assert stored_inferred.height == len(inferred_mappings)
assert stored_memberships.height == len(memberships)
assert set(stored_ttype["target_cluster"]) <= set(ttype_parent)
assert set(stored_inferred["target_cluster"]) <= set(mettype_parent)
assert set(stored_memberships["cluster"]) <= set(mettype_parent)
assert set(stored_inferred["source_cell"]).isdisjoint(stored_memberships["item"])
print("Excitatory Patch-seq CCM records verified.")